# Model data preparation
Model goal: classify building footprints as houses or not

Target classes:
- 0 - not a house
- 1 - house

Inputs:
- area
- type of nearest road
- distance to nearest road
- no. of buildings within 50 meters (100 meters)
- no. of POIs within 50 meters (100 meters)
- population density of city

In [1]:
from sklearn.ensemble import RandomForestClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score, f1_score
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("..//data//modeling//dataset.csv")
data.head()

,name,type,area,fclass_road,dist_road,within100m_bldg,within100m_poi,within50m_bldg,within50m_poi,block_id,density
0,Greenbelt 3,retail,10668.128160,unclassified,45.147977,32,28,8,15,32,26591.0
1,St. Jerome Emiliani and Sta. Susana Parish,church,1643.460315,service,25.317576,11,2,5,2,52,14092.0
2,E-COM Building,retail,14503.703873,tertiary,46.331879,61,9,1,1,60,20548.0
3,Riverbanks Center,retail,40482.611241,tertiary,50.656247,9,7,1,3,60,20548.0
4,Farmers' Market,retail,13460.500086,service,60.947306,26,16,1,2,45,18981.0


- 0: major roads
- 1: primary roads
- 2: secondary roads
- 3: tertiary roads
- 4: smaller roads and pathways

In [3]:
def type_to_int(bldg_type):
    if bldg_type == "house":
        return 1
    else:
        return 0

data["target"] = data["type"].apply(type_to_int)

def road_to_int(road_fclass):
    if road_fclass in ["trunk","trunk_link","motorway","motorway_link"]:
        return 0
    elif road_fclass in ["primary","primary_link","busway"]:
        return 1
    elif road_fclass in ["secondary","secondary_link"]:
        return 2
    elif road_fclass in ["tertiary","tertiary_link","service","unclassified"]:
        return 3
    else:
        return 4

data["road_class"] = data["fclass_road"].apply(road_to_int)

In [4]:
data.describe()

,area,dist_road,within100m_bldg,within100m_poi,within50m_bldg,within50m_poi,block_id,density,target,road_class
count,142245.000000,1.422450e+05,142245.000000,142245.000000,142245.000000,142245.000000,142245.000000,142221.000000,142245.000000,142245.000000
mean,226.228902,1.955520e+01,236.188344,3.002306,66.684833,0.796197,38.391479,26658.291877,0.801814,3.623692
std,1097.118864,1.741964e+01,138.730495,5.697885,42.439868,2.013519,21.691138,8116.199374,0.398635,0.648922
min,0.567752,1.718698e-07,0.000000,0.000000,0.000000,0.000000,0.000000,14092.000000,0.000000,0.000000
25%,30.476235,9.135556e+00,126.000000,0.000000,32.000000,0.000000,19.000000,20548.000000,1.000000,3.000000
50%,60.514826,1.461403e+01,222.000000,1.000000,63.000000,0.000000,40.000000,26845.000000,1.000000,4.000000
75%,153.120106,2.380260e+01,329.000000,4.000000,94.000000,1.000000,59.000000,27243.000000,1.000000,4.000000
max,156842.065112,2.682874e+02,897.000000,235.000000,255.000000,108.000000,81.000000,45326.000000,1.000000,4.000000


**Training-testing split strategy**

NCR's extent is divided into 0.03 deg x 0.03 deg rectangular grids (blocks).

Each block is either treated as part of the training or testing data.

In [5]:
y_col = "target"
x_cols = ["area","road_class","dist_road","within100m_bldg","within100m_poi","within50m_bldg","within50m_poi"]
block_train, block_test = train_test_split(range(81+1),random_state=1,test_size=0.2)

X_train = data.loc[data["block_id"].isin(block_train), x_cols].reset_index(drop=True)
y_train = data.loc[data["block_id"].isin(block_train), y_col].reset_index(drop=True)

X_test = data.loc[data["block_id"].isin(block_test), x_cols].reset_index(drop=True)
y_test = data.loc[data["block_id"].isin(block_test), y_col].reset_index(drop=True)

In [6]:
class_weight = {i:len(y_train)/np.sum(y_train==i) for i in range(2)}

rfmodel = RandomForestClassifier(n_estimators=100,random_state=1,class_weight=class_weight)
rfmodel.fit(X_train,y_train)

y_trainpred = rfmodel.predict(X_train)
y_testpred = rfmodel.predict(X_test)

In [7]:
print("Training:")
print(confusion_matrix(y_train,y_trainpred))
print("Accuracy score:",accuracy_score(y_train,y_trainpred))
print("Balanced accuracy score:",balanced_accuracy_score(y_train,y_trainpred,adjusted=False))
print("f1 score:",f1_score(y_train,y_trainpred))

print("Testing")
print(confusion_matrix(y_test,y_testpred))
print("Accuracy score:",accuracy_score(y_test,y_testpred))
print("Balanced accuracy score:",balanced_accuracy_score(y_test,y_testpred,adjusted=False))
print("f1 score:",f1_score(y_test,y_testpred))

Training:
[[23226     4]
 [    6 82310]]
Accuracy score: 0.999905254580941
Balanced accuracy score: 0.9998774595143434
f1 score: 0.9999392577294539
Testing
[[ 3607  1354]
 [ 1173 30565]]
Accuracy score: 0.9311425379438132
Balanced accuracy score: 0.8450561522099358
f1 score: 0.9603028732111158


In [9]:
import pickle

with open("..//data//modeling//rfmodel.pkl","wb") as f:
    pickle.dump(rfmodel,f)